In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

if os.path.basename(os.getcwd()) == 'Core':
    os.chdir('../../')

sns.set_theme(style="whitegrid")

def extrair_dados():
    res_base_path = "../resultados"
    modelos = ["CNN", "CRNN", "RESNET"]
    sementes = [7, 42, 123, 999, 1337]
    
    pastas = sorted([int(d.split('_')[-1]) for d in os.listdir(res_base_path) if d.startswith("n_mfccs_")])
    
    val_data = {m: {met: {n: [] for n in pastas} for met in ["accuracy", "f1-score", "precision", "recall"]} for m in modelos}
    gap_data = {m: {n: [] for n in pastas} for m in modelos}
    tempo_data = {m: {n: [] for n in pastas} for m in modelos}

    for n in pastas:
        for m in modelos:
            for s in sementes:
                path = os.path.join(res_base_path, f"n_mfccs_{n}", f"seed_{s}", f"{m}_history.pkl")
                if os.path.exists(path):
                    with open(path, 'rb') as f:
                        h = pickle.load(f)
                        val_data[m]["accuracy"][n].append(h.get('val_acc', 0))
                        val_data[m]["f1-score"][n].append(h.get('val_f1', 0))
                        val_data[m]["precision"][n].append(h.get('val_precision', 0))
                        val_data[m]["recall"][n].append(h.get('val_recall', 0))
                        
                        t_acc = h.get('train_acc', 0)
                        v_acc = h.get('val_acc', 0)
                        gap_data[m][n].append(t_acc - v_acc)

    tempo_path = os.path.join(res_base_path, "tempos_execucao.pkl")
    if os.path.exists(tempo_path):
        with open(tempo_path, 'rb') as f:
            t_dict = pickle.load(f)
            for m in modelos:
                if m in t_dict:
                    for n in pastas:
                        if n in t_dict[m]:
                            tempo_data[m][n] = list(t_dict[m][n].values())

    return val_data, gap_data, tempo_data, pastas, modelos

def plotar_benchmark(val_data, gap_data, tempo_data, pastas, modelos):
    fig, axes = plt.subplots(6, 1, figsize=(12, 35))
    cores = {"CNN": "#1f77b4", "CRNN": "#2ca02c", "RESNET": "#d62728"}
    
    metrics = ["accuracy", "f1-score", "precision", "recall"]
    for idx, met in enumerate(metrics):
        ax = axes[idx]
        for m in modelos:
            medias = [np.mean(val_data[m][met][n]) if val_data[m][met][n] else 0 for n in pastas]
            stds = [np.std(val_data[m][met][n]) if val_data[m][met][n] else 0 for n in pastas]
            ax.plot(pastas, medias, label=m, color=cores[m], marker='o', linewidth=2)
            ax.fill_between(pastas, np.array(medias) - np.array(stds), 
                            np.array(medias) + np.array(stds), color=cores[m], alpha=0.1)
        ax.set_title(f"Validação: {met.upper()} (Média Global)", fontweight='bold')
        ax.set_xscale('log', base=2); ax.set_xticks(pastas); ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.legend()

    # Gráfico do GAP
    ax_gap = axes[4]
    for m in modelos:
        medias_gap = [np.mean(gap_data[m][n]) if gap_data[m][n] else 0 for n in pastas]
        ax_gap.plot(pastas, medias_gap, label=f"GAP {m}", color=cores[m], linestyle='--', marker='s')
    ax_gap.set_title("GAP de Overfitting Médio (Train Acc - Val Acc)", fontweight='bold', color='darkred')
    ax_gap.set_xscale('log', base=2); ax_gap.set_xticks(pastas); ax_gap.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax_gap.legend()

    # Gráfico de Tempo
    ax_tempo = axes[5]
    for m in modelos:
        medias_t = [np.mean(tempo_data[m][n])/60 if tempo_data[m][n] else 0 for n in pastas]
        ax_tempo.plot(pastas, medias_t, label=m, color=cores[m], marker='^')
    ax_tempo.set_title("Tempo Médio de Treino (Minutos)", fontweight='bold')
    ax_tempo.set_xscale('log', base=2); ax_tempo.set_xticks(pastas); ax_tempo.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax_tempo.legend()

    plt.tight_layout()
    plt.show()

v_d, g_d, t_d, p, m = extrair_dados()
plotar_benchmark(v_d, g_d, t_d, p, m)

ValueError: Data has no positive values, and therefore cannot be log-scaled.

Error in callback <function _draw_all_if_interactive at 0x770a410204a0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Data has no positive values, and therefore cannot be log-scaled.

ValueError: Data has no positive values, and therefore cannot be log-scaled.

<Figure size 1200x3500 with 6 Axes>